# Entraînement ACT (LeRobot) sur GPU

Équivalent de `make train` + `make push-checkpoint`, à exécuter sur le Jupyter du serveur GPU.

1. Remplir la cellule **Configuration**.
2. Exécuter les cellules dans l'ordre.
3. Côté robot : `make eval HF_USER=<user> TASK=<tache>` (charge le modèle depuis le Hub).

## 1. Configuration

In [ ]:
HF_USER = "edepoix"        # compte Hugging Face
TASK = "test_wrist"        # nom du dataset (= nom du modèle poussé)
LEROBOT_VERSION = "0.5.1"  # même version que le conteneur (pyproject.toml)

STEPS = 100_000            # défaut lerobot ; 20k-50k suffisent souvent pour un petit dataset
BATCH_SIZE = 8
SAVE_FREQ = 20_000
NUM_WORKERS = 4
PUSH_TO_HUB = True         # pousse le modèle sur HF_USER/TASK à la fin

OUTPUT_DIR = f"outputs/{TASK}"
DATASET_REPO = f"{HF_USER}/{TASK}"

## 2. Vérification du GPU et de Python

LeRobot 0.5.1 demande Python 3.12.

In [ ]:
import sys
print("Python", sys.version)
!nvidia-smi

## 3. Installation de LeRobot

In [ ]:
!pip install -q "lerobot=={LEROBOT_VERSION}"
import lerobot, torch
print("lerobot", lerobot.__version__)
print("torch", torch.__version__, "| CUDA dispo :", torch.cuda.is_available())
assert torch.cuda.is_available(), "Pas de GPU visible : vérifier le noyau / l'installation de torch"

## 4. Connexion Hugging Face

Utiliser un token avec droit d'écriture. Ne pas l'écrire en clair dans le notebook.

In [ ]:
import os
from getpass import getpass
from huggingface_hub import login, whoami

token = os.environ.get("HF_TOKEN") or getpass("Token Hugging Face : ")
login(token=token)
print("Connecté en tant que", whoami()["name"])

## 5. Vérification du dataset

In [ ]:
from huggingface_hub import list_repo_files
files = list_repo_files(DATASET_REPO, repo_type="dataset")
print(len(files), "fichiers dans", DATASET_REPO)
print("\n".join(f for f in files if f.startswith("meta/")))

## 6. Entraînement

Pour reprendre un run interrompu : remplacer la commande par `!lerobot-train --config_path={OUTPUT_DIR}/checkpoints/last/pretrained_model/train_config.json --resume=true`.

In [ ]:
!lerobot-train \
  --policy.type=act \
  --policy.device=cuda \
  --policy.push_to_hub=false \
  --dataset.repo_id={DATASET_REPO} \
  --output_dir={OUTPUT_DIR} \
  --job_name=act_{TASK} \
  --steps={STEPS} \
  --batch_size={BATCH_SIZE} \
  --save_freq={SAVE_FREQ} \
  --num_workers={NUM_WORKERS} \
  --wandb.enable=false

## 7. Envoi du modèle sur le Hub

Même chose que `make push-checkpoint` : envoie `checkpoints/last/pretrained_model` vers `HF_USER/TASK`.

In [ ]:
from huggingface_hub import HfApi

ckpt = f"{OUTPUT_DIR}/checkpoints/last/pretrained_model"
print(os.listdir(ckpt))
if PUSH_TO_HUB:
    api = HfApi()
    api.create_repo(f"{HF_USER}/{TASK}", repo_type="model", exist_ok=True)
    api.upload_folder(folder_path=ckpt, repo_id=f"{HF_USER}/{TASK}", repo_type="model")
    print(f"Modèle disponible : https://huggingface.co/{HF_USER}/{TASK}")

## 8. (Optionnel) Archive à télécharger

Si le Hub n'est pas utilisé : télécharger l'archive puis la décompresser dans `outputs/<tache>/checkpoints/last/pretrained_model/` côté robot, et lancer `make eval TASK=<tache>` sans `HF_USER`.

In [ ]:
import shutil
archive = shutil.make_archive(f"{TASK}_pretrained_model", "zip", ckpt)
print("Archive :", archive)